# 🏁 Final Model Training & Deployment Pipeline
**Semi-Automated Classification Workflow**

Supported use cases:
- Customer Churn Prediction
- Credit Risk Prediction
- Loan Default Prediction

| Section | Purpose |
|---|---|
| **Section 1 — Final Model Verification** | Train the final LightGBM model and verify performance |
| **Section 2 — Deployment Pipeline** | Build and save a production-ready pipeline |
| **Section 3 — Interactive Pipeline Test** | Simulate Streamlit predictions inside Colab |

---

# Section 1 — Final Model Verification

## 1.1 — Import Libraries

Install and load required libraries.

In [ ]:
!pip install -q lightgbm imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, ConfusionMatrixDisplay,
                              classification_report)

from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully.')

## 1.2 — Upload Dataset

Upload the cleaned dataset from the previous notebook.

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print(f'File loaded: {filename}')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')

## 1.3 — Select Features

Define the selected features and target column.

In [ ]:
selected_features = [
    'InternetService',
    'PaymentMethod',
    'tenure',
    'Contract',
    'MonthlyCharges',
    'TotalCharges',
    'MultipleLines',
    'StreamingTV',
    'StreamingMovies',
    'PhoneService'
]

target_column = 'Churn'

print(f'Features ({len(selected_features)}): {selected_features}')
print(f'Target: {target_column}')

## 1.4 — Split Features and Target

Create X and y.

In [ ]:
X = df[selected_features].copy()
y = df[target_column].copy()

print(f'X Shape: {X.shape}')
print(f'y Shape: {y.shape}')

## 1.5 — Encode Categorical Features

Binary categories → Label Encoding. Multi-category columns → One-Hot Encoding.

In [ ]:
le = LabelEncoder()

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
binary_cols = [col for col in cat_cols if X[col].nunique() == 2]
multi_cols  = [col for col in cat_cols if X[col].nunique() > 2]

for col in binary_cols:
    X[col] = le.fit_transform(X[col].astype(str))

if multi_cols:
    X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

if y.dtype == 'object':
    y = pd.Series(le.fit_transform(y.astype(str)))

total_encoded = X.shape[1]

print(f'Label Encoded    : {binary_cols}')
print(f'One-Hot Encoded  : {multi_cols}')
print(f'Total Features After Encoding: {total_encoded}')

## 1.6 — Train Test Split

Split into 80% training and 20% testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train Shape : {X_train.shape}')
print(f'Test Shape  : {X_test.shape}')

## 1.7 — Check Target Balance

Display the class distribution of the training data.

In [ ]:
counts = y_train.value_counts()
percentages = y_train.value_counts(normalize=True) * 100

display(pd.DataFrame({'Class': counts.index, 'Count': counts.values, 'Percentage (%)': percentages.values}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=y_train, ax=ax, palette='Set2')
ax.set_title('Training Set Class Distribution')
plt.show()

IS_IMBALANCED = percentages.min() < 30

## 1.8 — Apply SMOTE

Apply SMOTE to training data only if the dataset is imbalanced.

In [ ]:
smote_applied = 'No'

if IS_IMBALANCED:
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    y_train = pd.Series(y_train)
    smote_applied = 'Yes'
    print(f'SMOTE applied. New training shape: {X_train.shape}')
else:
    print('SMOTE skipped.')

## 1.9 — Train Final LightGBM

Train the final model using the recommended parameters.

In [ ]:
final_model = LGBMClassifier(
    learning_rate=0.01,
    n_estimators=500,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0,
    reg_lambda=0,
    random_state=42,
    verbose=-1
)

final_model.fit(X_train, y_train)

print('Final model trained successfully.')

## 1.10 — Evaluate Model

Evaluate the final model on the test set.

In [ ]:
y_pred  = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

v_accuracy  = accuracy_score(y_test, y_pred)
v_precision = precision_score(y_test, y_pred)
v_recall    = recall_score(y_test, y_pred)
v_f1        = f1_score(y_test, y_pred)
v_roc_auc   = roc_auc_score(y_test, y_proba)

print(f'Accuracy  : {v_accuracy:.4f}')
print(f'Precision : {v_precision:.4f}')
print(f'Recall    : {v_recall:.4f}')
print(f'F1 Score  : {v_f1:.4f}')
print(f'ROC AUC   : {v_roc_auc:.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, label=f'ROC AUC = {v_roc_auc:.4f}')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
plt.show()

## 1.11 — Final Verification Summary

Summary of the model verification results.

In [ ]:
print('=' * 52)
print('       FINAL VERIFICATION SUMMARY')
print('=' * 52)
print(f'Dataset Shape     : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Selected Features : {len(selected_features)}')
print(f'Encoding Applied  : Label + One-Hot')
print(f'SMOTE Applied     : {smote_applied}')
print(f'Model             : LightGBM')
print()
print(f'Accuracy  : {v_accuracy:.4f}')
print(f'Precision : {v_precision:.4f}')
print(f'Recall    : {v_recall:.4f}')
print(f'F1 Score  : {v_f1:.4f}')
print(f'ROC AUC   : {v_roc_auc:.4f}')
print('=' * 52)

---
# Section 2 — Deployment Pipeline

> This section is **completely independent** from Section 1.
> It reloads the original cleaned dataset — no SMOTE, no manual encoding.
> The pipeline handles all preprocessing internally.

## 2.1 — Reload Dataset

Load the cleaned dataset into a fresh dataframe.

In [ ]:
data = pd.read_csv(filename)

print(f'Dataset reloaded: {data.shape[0]} rows x {data.shape[1]} columns')

## 2.2 — Select Features

Use the same selected features and target.

In [ ]:
X_dep = data[selected_features].copy()
y_dep = data[target_column].copy()

if y_dep.dtype == 'object':
    y_dep = pd.Series(LabelEncoder().fit_transform(y_dep.astype(str)))

print(f'X Shape: {X_dep.shape}')
print(f'y Shape: {y_dep.shape}')

## 2.3 — Train Test Split

Split the original dataset. No SMOTE applied.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dep, y_dep, test_size=0.2, random_state=42, stratify=y_dep
)

print(f'Train Shape : {X_tr.shape}')
print(f'Test Shape  : {X_te.shape}')

## 2.4 — Build Preprocessing Pipeline

OneHotEncoder for categorical features. Numeric features pass through unchanged.

In [ ]:
pipe_cat_cols = X_dep.select_dtypes(include=['object', 'category']).columns.tolist()
pipe_num_cols = X_dep.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), pipe_cat_cols),
        ('num', 'passthrough', pipe_num_cols)
    ]
)

print(f'Categorical : {pipe_cat_cols}')
print(f'Numeric     : {pipe_num_cols}')

## 2.5 — Create LightGBM Model

Create the model using the same final parameters.

In [ ]:
pipe_model = LGBMClassifier(
    learning_rate=0.01,
    n_estimators=500,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0,
    reg_lambda=0,
    random_state=42,
    verbose=-1
)

print('Model created.')

## 2.6 — Build Machine Learning Pipeline

Bundle the preprocessor and model into one pipeline.

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', pipe_model)
])

print('Pipeline created.')
print(pipeline)

## 2.7 — Train Pipeline

Fit the pipeline on the original training data only.

In [ ]:
pipeline.fit(X_tr, y_tr)

print('Pipeline trained successfully.')

## 2.8 — Save Pipeline

Save and download the trained pipeline.

In [ ]:
joblib.dump(pipeline, 'lightgbm_pipeline.pkl')

print('Pipeline saved successfully.')

files.download('lightgbm_pipeline.pkl')

## 2.9 — Pipeline Summary

Summary of the deployment pipeline.

In [ ]:
print('=' * 52)
print('           PIPELINE SUMMARY')
print('=' * 52)
print('  Preprocessing:')
print(f'    One-Hot Encoding  : {pipe_cat_cols}')
print(f'    Numeric Pass-Through : {pipe_num_cols}')
print('  Model: LightGBM')
print()
print('  Pipeline File : lightgbm_pipeline.pkl')
print('=' * 52)

---
# Section 3 — Interactive Pipeline Test

> Verify the saved pipeline works correctly before deployment.
> This simulates the future Streamlit application — enter values and receive a prediction directly inside Colab.

## 3.1 — Load Pipeline

Load the saved pipeline from disk.

In [ ]:
loaded_pipeline = joblib.load('lightgbm_pipeline.pkl')

print('Pipeline loaded successfully.')

## 3.2 — Create Interactive Input Form

Build an input form using ipywidgets. Dropdown options are pulled directly from the dataset.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

w_internet = widgets.Dropdown(
    options=sorted(data['InternetService'].dropna().unique()),
    description='InternetService:'
)

w_payment = widgets.Dropdown(
    options=sorted(data['PaymentMethod'].dropna().unique()),
    description='PaymentMethod:'
)

w_contract = widgets.Dropdown(
    options=sorted(data['Contract'].dropna().unique()),
    description='Contract:'
)

w_multilines = widgets.Dropdown(
    options=sorted(data['MultipleLines'].dropna().unique()),
    description='MultipleLines:'
)

w_streamingtv = widgets.Dropdown(
    options=sorted(data['StreamingTV'].dropna().unique()),
    description='StreamingTV:'
)

w_streamingmovies = widgets.Dropdown(
    options=sorted(data['StreamingMovies'].dropna().unique()),
    description='StreamingMovies:'
)

w_phone = widgets.Dropdown(
    options=sorted(data['PhoneService'].dropna().unique()),
    description='PhoneService:'
)

w_tenure = widgets.IntText(value=12, description='tenure:')
w_monthly = widgets.FloatText(value=65.0, description='MonthlyCharges:')
w_total = widgets.FloatText(value=780.0, description='TotalCharges:')

display(
    w_internet, w_payment, w_contract, w_multilines,
    w_streamingtv, w_streamingmovies, w_phone,
    w_tenure, w_monthly, w_total
)

print('Form ready. Fill in the values above, then run the next cell.')

## 3.3 — Predict Button

Click the Predict button to collect input values and run the pipeline.

In [ ]:
predict_button = widgets.Button(
    description='Predict',
    button_style='primary',
    layout=widgets.Layout(width='150px', height='40px')
)

output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()

        input_data = pd.DataFrame([{
            'InternetService':  w_internet.value,
            'PaymentMethod':    w_payment.value,
            'tenure':           w_tenure.value,
            'Contract':         w_contract.value,
            'MonthlyCharges':   w_monthly.value,
            'TotalCharges':     w_total.value,
            'MultipleLines':    w_multilines.value,
            'StreamingTV':      w_streamingtv.value,
            'StreamingMovies':  w_streamingmovies.value,
            'PhoneService':     w_phone.value
        }])

        prediction = loaded_pipeline.predict(input_data)[0]
        probability = loaded_pipeline.predict_proba(input_data)[0][1]

        label = 'Likely Churn' if prediction == 1 else 'Not Likely to Churn'

        print('=' * 40)
        print('          PREDICTION RESULT')
        print('=' * 40)
        print(f'  Prediction  : {label}')
        print(f'  Probability : {probability * 100:.2f}%')
        print('=' * 40)

predict_button.on_click(on_predict_clicked)

display(predict_button, output)

---
## 3.4 — Learning Notes

- The pipeline **automatically performs preprocessing** before every prediction — no manual encoding needed.
- The pipeline **automatically encodes categorical values** using the same rules learned during training.
- The user only enters **raw information** — exactly like a Streamlit user would.
- This is the **same workflow** that will be used in the final Streamlit deployment app.

```python
# In Streamlit:
pipeline = joblib.load('lightgbm_pipeline.pkl')
prediction = pipeline.predict(input_df)
probability = pipeline.predict_proba(input_df)[0][1]
```

✅ The pipeline is ready for deployment.